# Data Cleaning

This notebook cleans and prepares the raw retail sales datasets for analysis.

## Objectives

- Identify data quality issues
- Handle missing values
- Detect duplicates
- Correct data types
- Review outliers
- Standardize inconsistent values

In [1]:
import pandas as pd
import numpy as np

regions = pd.read_csv("../../data/raw/Regions.csv")
products = pd.read_csv("../../data/raw/Products.csv")
customers = pd.read_csv("../../data/raw/Customers.csv")
orders = pd.read_csv("../../data/raw/Orders.csv")
order_items = pd.read_csv("../../data/raw/OrderItems.csv")

In [2]:
customers.isnull().sum()

CustomerID           0
CustomerName         0
Gender               0
City                 0
RegionID             0
CustomerSegment      0
RegistrationDate     0
Email               59
Phone               36
IsActive             0
dtype: int64

## Missing Values

Identify and evaluate missing values before deciding how they should be handled.

In [3]:
customers[customers[["Email", "Phone"]].isnull().any(axis=1)].head(10)

,CustomerID,CustomerName,Gender,City,RegionID,CustomerSegment,RegistrationDate,Email,Phone,IsActive
33,34,شهد العنزي,Female,مكة,2,Premium,2025-05-17,customer34@example.com,NaN,1
58,59,فهد الشمري,Male,بريدة,5,Regular,2025-02-24,NaN,561167978.0,1
76,77,لطيفة الحربي,Female,عنيزة,5,Regular,2024-05-23,NaN,576118957.0,1
82,83,خالد العنزي,Male,الطائف,2,Premium,2024-04-22,customer83@example.com,NaN,0
98,99,نورة الشمري,Female,الخبر,3,Corporate,2024-02-26,NaN,538351344.0,1
101,102,لطيفة السبيعي,Female,الخبر,3,Regular,2025-02-18,NaN,592963368.0,1
123,124,شهد المالكي,Female,جدة,2,Premium,2025-04-25,customer124@example.com,NaN,0
126,127,عبدالعزيز العنزي,Male,بريدة,5,Regular,2024-05-27,customer127@example.com,NaN,1
128,129,تركي الزهراني,Male,الرياض,1,Regular,2025-01-17,customer129@example.com,NaN,1
137,138,أحمد الدوسري,Male,بريدة,5,Premium,2025-03-24,NaN,596592295.0,1


In [4]:
customers[["Email", "Phone"]].isnull().sum()

Email    59
Phone    36
dtype: int64

In [5]:
customers[
    customers["Email"].isnull() &
    customers["Phone"].isnull()
].shape[0]

1

### Missing Values Decision

Missing values were found only in `Email` and `Phone`.

These fields are not required for sales analysis, so customer records will be retained without deleting or imputing synthetic contact information.

In [6]:
print("Regions:")
print(regions.isnull().sum())

print("\nProducts:")
print(products.isnull().sum())

print("\nOrders:")
print(orders.isnull().sum())

print("\nOrder Items:")
print(order_items.isnull().sum())

Regions:
RegionID      0
RegionName    0
dtype: int64

Products:
ProductID      0
ProductName    0
Category       0
SubCategory    0
UnitPrice      0
UnitCost       0
IsActive       0
dtype: int64

Orders:
OrderID           0
CustomerID        0
OrderDate         0
OrderStatus       0
PaymentMethod     0
ShippingMethod    0
dtype: int64

Order Items:
OrderItemID        0
OrderID            0
ProductID          0
Quantity           0
UnitPrice          0
DiscountPercent    0
dtype: int64


 هذا يفحص التكرار الكامل للصف بالكامل.

In [7]:
customers.duplicated().sum()

np.int64(0)

 الآن نكمل Duplicates بشكل أدق، لأن التكرار مو لازم يكون الصف كامل مكرر. ممكن مثلًا CustomerID يتكرر مع بيانات مختلفة. إذا طلع 0 فهذا يعني أن مفتاح العميل فريد مثل المتوقع.

In [8]:
customers["CustomerID"].duplicated().sum()

np.int64(0)

 بعدها شغّل نفس الفكرة على المفاتيح الأساسية للجداول كلها
 إذا كلها أصفار، نكون تأكدنا أن المفاتيح الأساسية غير مكررة.

In [9]:
print("RegionID duplicates:", regions["RegionID"].duplicated().sum())
print("ProductID duplicates:", products["ProductID"].duplicated().sum())
print("CustomerID duplicates:", customers["CustomerID"].duplicated().sum())
print("OrderID duplicates:", orders["OrderID"].duplicated().sum())
print("OrderItemID duplicates:", order_items["OrderItemID"].duplicated().sum())

RegionID duplicates: 0
ProductID duplicates: 0
CustomerID duplicates: 0
OrderID duplicates: 0
OrderItemID duplicates: 0



## هذا يعرض العملاء اللي أسماؤهم متكررة، حتى لو باقي البيانات مختلفة

In [10]:
customers[customers.duplicated(subset=["CustomerName"], keep=False)] \
    .sort_values("CustomerName") \
    .head(20)

,CustomerID,CustomerName,Gender,City,RegionID,CustomerSegment,RegistrationDate,Email,Phone,IsActive
265,266,أحمد الحربي,Male,مكة,2,Regular,2024-03-02,customer266@example.com,548451741.0,1
464,465,أحمد الحربي,Male,الخبر,3,Regular,2025-04-26,customer465@example.com,543565097.0,1
306,307,أحمد الحربي,Male,الرياض,1,Regular,2024-04-17,customer307@example.com,560393104.0,1
711,712,أحمد الحربي,Male,المدينة المنورة,4,Regular,2024-11-05,customer712@example.com,527748717.0,1
527,528,أحمد الدوسري,Male,جدة,2,Regular,2024-12-23,NaN,599522760.0,1
381,382,أحمد الدوسري,Male,الدمام,3,Regular,2024-11-04,customer382@example.com,558061434.0,0
137,138,أحمد الدوسري,Male,بريدة,5,Premium,2025-03-24,NaN,596592295.0,1
952,953,أحمد الدوسري,Male,مكة,2,Regular,2024-09-15,customer953@example.com,585208373.0,1
777,778,أحمد الرشيدي,Male,الدمام,3,Regular,2024-02-29,customer778@example.com,529120731.0,1
884,885,أحمد الرشيدي,Male,الخرج,1,Premium,2024-10-13,customer885@example.com,543358157.0,1


## وهذا يفحص هل نفس البريد مستخدم لأكثر من عميل.

In [11]:
customers[customers.duplicated(subset=["Email"], keep=False) & customers["Email"].notna()] \
    .sort_values("Email") \
    .head(20)

,CustomerID,CustomerName,Gender,City,RegionID,CustomerSegment,RegistrationDate,Email,Phone,IsActive
0,1,ناصر العتيبي,Male,الدمام,3,Regular,2024-05-31,customer1@example.com,558942697.0,1
999,1000,ناصر العتيبي,Male,الدمام,3,Regular,2024-05-31,customer1@example.com,558942697.0,1
1,2,فيصل الشمري,Male,عنيزة,5,Regular,2024-04-15,customer2@example.com,564543049.0,1
998,999,فيصل الشمري,Male,عنيزة,5,Regular,2024-04-15,customer2@example.com,564543049.0,1
2,3,عبدالله الدوسري,Male,جدة,2,Premium,2024-01-26,customer3@example.com,565259205.0,1
997,998,عبدالله الدوسري,Male,جدة,2,Premium,2024-01-26,customer3@example.com,565259205.0,1
3,4,سعود الدوسري,Male,الخرج,1,Regular,2024-02-09,customer4@example.com,571780854.0,1
996,997,سعود الدوسري,Male,الخرج,1,Regular,2024-02-09,customer4@example.com,571780854.0,1
4,5,عبدالله العتيبي,Male,عنيزة,5,Regular,2024-12-02,customer5@example.com,547463522.0,1
995,996,عبدالله العتيبي,Male,عنيزة,5,Regular,2024-12-02,customer5@example.com,547463522.0,1


## الفكرة هنا مهمة: Duplicates ليست فقط صفوفًا متطابقة. أحيانًا يكون عندك سجلان مختلفان لكن يمثلان نفس الشخص.

In [12]:
customers[customers.duplicated(subset=["Phone"], keep=False) & customers["Phone"].notna()] \
    .sort_values("Phone") \
    .head(20)

,CustomerID,CustomerName,Gender,City,RegionID,CustomerSegment,RegistrationDate,Email,Phone,IsActive
995,996,عبدالله العتيبي,Male,عنيزة,5,Regular,2024-12-02,customer5@example.com,547463522.0,1
4,5,عبدالله العتيبي,Male,عنيزة,5,Regular,2024-12-02,customer5@example.com,547463522.0,1
0,1,ناصر العتيبي,Male,الدمام,3,Regular,2024-05-31,customer1@example.com,558942697.0,1
999,1000,ناصر العتيبي,Male,الدمام,3,Regular,2024-05-31,customer1@example.com,558942697.0,1
998,999,فيصل الشمري,Male,عنيزة,5,Regular,2024-04-15,customer2@example.com,564543049.0,1
1,2,فيصل الشمري,Male,عنيزة,5,Regular,2024-04-15,customer2@example.com,564543049.0,1
997,998,عبدالله الدوسري,Male,جدة,2,Premium,2024-01-26,customer3@example.com,565259205.0,1
2,3,عبدالله الدوسري,Male,جدة,2,Premium,2024-01-26,customer3@example.com,565259205.0,1
996,997,سعود الدوسري,Male,الخرج,1,Regular,2024-02-09,customer4@example.com,571780854.0,1
3,4,سعود الدوسري,Male,الخرج,1,Regular,2024-02-09,customer4@example.com,571780854.0,1


## نسوي نفس الاستعلام بس هنا استبعدنا فقط:CustomerID 
## وسألنا Pandas: هل توجد سجلات متطابقة في بيانات العميل نفسها، حتى لو كان لكل واحد CustomerID مختلف؟

In [13]:
duplicate_customers = customers[
    customers.duplicated(
        subset=[
            "CustomerName",
            "Gender",
            "City",
            "RegionID",
            "CustomerSegment",
            "RegistrationDate",
            "Email",
            "Phone",
            "IsActive"
        ],
        keep=False
    )
]

duplicate_customers.sort_values(["CustomerName", "Email"])

,CustomerID,CustomerName,Gender,City,RegionID,CustomerSegment,RegistrationDate,Email,Phone,IsActive
3,4,سعود الدوسري,Male,الخرج,1,Regular,2024-02-09,customer4@example.com,571780854.0,1
996,997,سعود الدوسري,Male,الخرج,1,Regular,2024-02-09,customer4@example.com,571780854.0,1
2,3,عبدالله الدوسري,Male,جدة,2,Premium,2024-01-26,customer3@example.com,565259205.0,1
997,998,عبدالله الدوسري,Male,جدة,2,Premium,2024-01-26,customer3@example.com,565259205.0,1
4,5,عبدالله العتيبي,Male,عنيزة,5,Regular,2024-12-02,customer5@example.com,547463522.0,1
995,996,عبدالله العتيبي,Male,عنيزة,5,Regular,2024-12-02,customer5@example.com,547463522.0,1
1,2,فيصل الشمري,Male,عنيزة,5,Regular,2024-04-15,customer2@example.com,564543049.0,1
998,999,فيصل الشمري,Male,عنيزة,5,Regular,2024-04-15,customer2@example.com,564543049.0,1
0,1,ناصر العتيبي,Male,الدمام,3,Regular,2024-05-31,customer1@example.com,558942697.0,1
999,1000,ناصر العتيبي,Male,الدمام,3,Regular,2024-05-31,customer1@example.com,558942697.0,1


## (10, 10) معناها:
## عندنا 10 سجلات عملاء داخلة ضمن حالات التكرار المنطقي.
## وكل سجل فيه 10 أعمدة.
## وطبعا هذا لا يعني عندنا 10 عملاء مكررين إضافيين. غالبًا عندنا 5 أزواج، كل زوج يمثل نفس العميل مرتين.

In [14]:
duplicate_customers.shape

(10, 10)

## هدفنا هنا بسيط:
## نعرف هل الـCustomerID الأصلي والمكرر عندهم Orders فعلًا أم لا.

In [15]:
duplicate_ids = duplicate_customers["CustomerID"].tolist()

orders[
    orders["CustomerID"].isin(duplicate_ids)
].sort_values("CustomerID")

,OrderID,CustomerID,OrderDate,OrderStatus,PaymentMethod,ShippingMethod
181,182,1,2024-06-05,Completed,Card,Delivery
1455,1456,1,2025-01-23,Completed,Digital Wallet,Delivery
1500,1501,1,2025-02-12,Completed,Cash,Delivery
2697,2698,1,2024-09-25,Returned,Bank Transfer,Pickup
2262,2263,1,2025-07-23,Cancelled,Cash,Pickup
2067,2068,1,2024-05-20,Completed,Cash,Delivery
1885,1886,1,2024-03-25,Completed,Cash,Delivery
1843,1844,1,2025-12-28,Completed,Card,Delivery
4879,4880,1,2024-08-24,Completed,Cash,Delivery
2957,2958,1,2024-03-07,Completed,Cash,Delivery


## هذا يعطينا عدد الطلبات لكل CustomerID من العملاء المكررين.

In [16]:
orders[
    orders["CustomerID"].isin(duplicate_ids)
]["CustomerID"].value_counts().sort_index()

CustomerID
1       10
2        2
3        8
4        6
5        6
996      5
997      3
998      5
999      3
1000     2
Name: count, dtype: int64

## الهدف من هذه الخطوة أننا نبني Duplicate Groups واضحة بدل ما نعتمد على النظر فقط.
## نثبت الأزواج رسميًا
## ولو نلاحظ الحين 
## ثبتنا  أن عندنا 5 مجموعات تكرار، وكل مجموعة فيها سجلين لنفس العميل

In [17]:
match_columns = [
    "CustomerName",
    "Gender",
    "City",
    "RegionID",
    "CustomerSegment",
    "RegistrationDate",
    "Email",
    "Phone",
    "IsActive"
]

duplicate_groups = (
    duplicate_customers
    .groupby(match_columns, dropna=False)["CustomerID"]
    .agg(list)
    .reset_index(name="CustomerIDs")
)

duplicate_groups[
    ["CustomerName", "Email", "Phone", "CustomerIDs"]
]

,CustomerName,Email,Phone,CustomerIDs
0,سعود الدوسري,customer4@example.com,571780854.0,"[4, 997]"
1,عبدالله الدوسري,customer3@example.com,565259205.0,"[3, 998]"
2,عبدالله العتيبي,customer5@example.com,547463522.0,"[5, 996]"
3,فيصل الشمري,customer2@example.com,564543049.0,"[2, 999]"
4,ناصر العتيبي,customer1@example.com,558942697.0,"[1, 1000]"


### Duplicate Customer Resolution

Duplicate customer records were identified using matching customer attributes.

The lower `CustomerID` is retained as the canonical record, and orders linked to duplicate IDs are reassigned before removing duplicate customer rows.

# هذه مجرد خريطة تحويل، وما عدلنا أي بيانات للحين

In [18]:
customer_id_map = {
    997: 4,
    998: 3,
    996: 5,
    999: 2,
    1000: 1
}

customer_id_map

{997: 4, 998: 3, 996: 5, 999: 2, 1000: 1}

## نطبق التحويل على جدول orders، لكن بشكل منضبط ونتحقق قبل وبعد.
## فقط غيرنا CustomerID لبعض الطلبات حتى ترتبط بالسجل الأصلي الصحيح.

In [19]:
orders_before = len(orders)

orders["CustomerID"] = orders["CustomerID"].replace(customer_id_map)

orders_after = len(orders)

print("Orders before:", orders_before)
print("Orders after:", orders_after)

Orders before: 5000
Orders after: 5000


## بعدها لازم حقق أن الـIDs المكررة لم تعد مستخدمة داخل orders: لذا مهم يكون النتيجة فارغة

In [20]:
orders[
    orders["CustomerID"].isin(customer_id_map.keys())
]["CustomerID"].value_counts()

Series([], Name: count, dtype: int64)

## اخر شي نسويه  حذف سجلات العملاء المكررة من customers:

In [21]:
customers = customers[
    ~customers["CustomerID"].isin(customer_id_map.keys())
].copy()

##  ولو نلاحظ تغير القيمة من (1000, 10) الى (995,10)وبعدها تأكد

In [22]:
customers.shape

(995, 10)

## وبعد هذي الخطوة نقدر نقول اننا خلصنا موضوع التكرار Duplicates  والاهم انه حافظت على علاقات الجداول يعني ما حذفت شي بدون ما اتاكد

## Data Type Conversion

Convert date columns to proper datetime types before analysis.

In [23]:
customers.dtypes

CustomerID            int64
CustomerName         object
Gender               object
City                 object
RegionID              int64
CustomerSegment      object
RegistrationDate     object
Email                object
Phone               float64
IsActive              int64
dtype: object

In [24]:
orders.dtypes

OrderID            int64
CustomerID         int64
OrderDate         object
OrderStatus       object
PaymentMethod     object
ShippingMethod    object
dtype: object

In [25]:
customers["RegistrationDate"] = pd.to_datetime(customers["RegistrationDate"])

orders["OrderDate"] = pd.to_datetime(orders["OrderDate"])

In [26]:
print(customers["RegistrationDate"].dtype)
print(orders["OrderDate"].dtype)

datetime64[ns]
datetime64[ns]


In [27]:
customers["Phone"] = customers["Phone"].astype("Int64").astype("string")

In [28]:
customers["Phone"].dtype

string[python]

In [29]:
customers[["Phone"]].head(10)

,Phone
0,558942697
1,564543049
2,565259205
3,571780854
4,547463522
5,587925434
6,525521714
7,599152472
8,584411983
9,583542887


In [30]:
customers["IsActive"].value_counts()

IsActive
1    904
0     91
Name: count, dtype: int64

In [31]:
products["IsActive"].value_counts()

IsActive
1    93
0     7
Name: count, dtype: int64

In [32]:
customers["IsActive"] = customers["IsActive"].astype(bool)
products["IsActive"] = products["IsActive"].astype(bool)

In [33]:
customers["IsActive"] = customers["IsActive"].astype(bool)
products["IsActive"] = products["IsActive"].astype(bool)

In [34]:
print("Customers:")
print(customers.dtypes)

print("\nProducts:")
print(products.dtypes)

print("\nOrders:")
print(orders.dtypes)

Customers:
CustomerID                   int64
CustomerName                object
Gender                      object
City                        object
RegionID                     int64
CustomerSegment             object
RegistrationDate    datetime64[ns]
Email                       object
Phone               string[python]
IsActive                      bool
dtype: object

Products:
ProductID        int64
ProductName     object
Category        object
SubCategory     object
UnitPrice      float64
UnitCost       float64
IsActive          bool
dtype: object

Orders:
OrderID                    int64
CustomerID                 int64
OrderDate         datetime64[ns]
OrderStatus               object
PaymentMethod             object
ShippingMethod            object
dtype: object


## Outlier Detection

Identify unusual values in numeric fields before deciding whether they should be retained, capped, or removed.

In [35]:
order_items["Quantity"].describe()

count    13362.000000
mean         1.929651
std          1.747428
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max         55.000000
Name: Quantity, dtype: float64

In [36]:
Q1 = order_items["Quantity"].quantile(0.25)
Q3 = order_items["Quantity"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

Q1: 1.0
Q3: 2.0
IQR: 1.0
Lower Bound: -0.5
Upper Bound: 3.5


In [37]:
quantity_outliers = order_items[
    (order_items["Quantity"] < lower_bound) |
    (order_items["Quantity"] > upper_bound)
]

quantity_outliers.shape

(1354, 6)

In [38]:
quantity_outliers["Quantity"].value_counts().sort_index()

Quantity
4     916
5     422
26      1
29      1
30      1
31      1
36      2
38      1
39      1
43      3
45      1
50      1
51      1
52      1
55      1
Name: count, dtype: int64

In [39]:
order_items["Quantity"].value_counts().sort_index()

Quantity
1     6621
2     3419
3     1968
4      916
5      422
26       1
29       1
30       1
31       1
36       2
38       1
39       1
43       3
45       1
50       1
51       1
52       1
55       1
Name: count, dtype: int64

In [40]:
order_items[order_items["Quantity"] > 5]["Quantity"].describe()

count    16.00000
mean     40.43750
std       8.84661
min      26.00000
25%      34.75000
50%      41.00000
75%      46.25000
max      55.00000
Name: Quantity, dtype: float64

### Quantity Outlier Decision

The IQR method flagged values above 3.5 as statistical outliers.

However, quantities of 4 and 5 are still reasonable in a retail context.

The data distribution shows a clear jump from 5 to 26, with only 16 records above 5. Therefore, values greater than 5 are treated as true data-quality outliers.

In [41]:
quantity_outliers = order_items[
    order_items["Quantity"] > 5
]

quantity_outliers

,OrderItemID,OrderID,ProductID,Quantity,UnitPrice,DiscountPercent
1901,1902,708,57,51,771.32,0
2259,2260,828,27,43,248.66,0
3209,3210,1191,68,43,2338.01,0
3271,3272,1213,72,36,1230.62,0
3435,3436,1279,18,45,1442.78,0
5122,5123,1900,94,26,610.76,0
5278,5279,1959,18,52,1442.78,20
5333,5334,1979,28,30,247.06,5
6186,6187,2302,47,36,624.38,10
6632,6633,2477,52,50,1438.67,5


In [42]:
order_items_before = len(order_items)

order_items = order_items[
    order_items["Quantity"] <= 5
].copy()

order_items_after = len(order_items)

print("Before:", order_items_before)
print("After:", order_items_after)
print("Removed:", order_items_before - order_items_after)

Before: 13362
After: 13346
Removed: 16


In [43]:
order_items["Quantity"].describe()

count    13346.000000
mean         1.883486
std          1.090503
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max          5.000000
Name: Quantity, dtype: float64

## Data Standardization

Standardize inconsistent categorical values to ensure that equivalent values are represented consistently.

نسوي الامر هذا عشان يشكل لنا اختلافات  في أسماء المدن.

In [44]:
customers["City"].value_counts().sort_index()

City
RIYADH               4
Riyadh               6
الخبر               86
الخرج               84
الدمام              82
الرياض              78
الطائف              96
الظهران            103
المدينة المنورة     83
بريدة               91
جدة                 89
عنيزة              106
مكة                 87
Name: count, dtype: int64

طبعا تعمدت قبل اني اسوي الرياض بالشكل هذا عشان نعرف نكتشف وبعدها نعدل ونسوي توحيد لهم

In [45]:
city_mapping = {
    "RIYADH": "الرياض",
    "Riyadh": "الرياض"
}

customers["City"] = customers["City"].replace(city_mapping)

In [46]:
customers["City"].value_counts().sort_index()

City
الخبر               86
الخرج               84
الدمام              82
الرياض              88
الطائف              96
الظهران            103
المدينة المنورة     83
بريدة               91
جدة                 89
عنيزة              106
مكة                 87
Name: count, dtype: int64

## Export Clean Datasets

Export the cleaned datasets to the processed data folder for use in EDA, statistics, and Power BI.

In [47]:
regions.to_csv("../../data/processed/Regions_clean.csv", index=False)
products.to_csv("../../data/processed/Products_clean.csv", index=False)
customers.to_csv("../../data/processed/Customers_clean.csv", index=False)
orders.to_csv("../../data/processed/Orders_clean.csv", index=False)
order_items.to_csv("../../data/processed/OrderItems_clean.csv", index=False)

In [48]:
import os

print(os.listdir("../../data/processed"))

['Customers_clean.csv', 'OrderItems_clean.csv', 'Orders_clean.csv', 'Products_clean.csv', 'Regions_clean.csv']


In [49]:
print("Regions:", regions.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)

Regions: (5, 2)
Products: (100, 7)
Customers: (995, 10)
Orders: (5000, 6)
Order Items: (13346, 6)
